# XLS-R full fine-tuning
based on https://huggingface.co/blog/fine-tune-xlsr-wav2vec2


In [1]:
%pip install -U pip
%pip install --no-cache-dir 'transformers==4.57.1' accelerate 'datasets[audio]' evaluate jiwer safetensors huggingface_hub tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 67.0 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 26.0.1
    Uninstalling pip-26.0.1:
      Successfully uninstalled pip-26.0.1
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 236.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 136.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 796.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 1.2 GB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 1.2 GB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 253.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 590.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.2/801.2 kB 192.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 296.6 MB/s  0:00:00
  Attempting uninstall: hugg

In [1]:
import os
import re
import json
import gc
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Union

import numpy as np
import pandas as pd
import torch
import jiwer

from tqdm.auto import tqdm
from datasets import load_dataset, Audio
from huggingface_hub import notebook_login, HfFolder, create_repo

from transformers import (Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor,
    Wav2Vec2ForCTC, TrainingArguments, Trainer, EarlyStoppingCallback, set_seed)

In [2]:
# notebook_login()

In [3]:
hf_token = HfFolder.get_token()

if hf_token is None:
    raise ValueError('HF token was not found. Run notebook_login() first.')

In [4]:
def create_xlsr_model(model_id):
    model = Wav2Vec2ForCTC.from_pretrained(
        model_id,
        attention_dropout=0.0,
        hidden_dropout=0.0,
        feat_proj_dropout=0.0,
        mask_time_prob=0.05,
        layerdrop=0.0,
        ctc_loss_reduction='mean',
        pad_token_id=processor.tokenizer.pad_token_id,
        vocab_size=len(processor.tokenizer),
        ignore_mismatched_sizes=True
    )

    model.freeze_feature_encoder()

    return model


def extract_all_chars(batch):
    all_text = ' '.join(batch['sentence'])
    vocab = sorted(list(set(all_text)))
    return {'vocab': [vocab], 'all_text': [all_text]}


def prepare_dataset(batch):
    audio = batch['audio']

    batch['input_values'] = processor(audio['array'], sampling_rate=audio['sampling_rate']).input_values[0]
    batch['input_length'] = len(batch['input_values'])
    batch['labels'] = processor(text=batch['sentence']).input_ids

    return batch


@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{'input_values': feature['input_values']} for feature in features]
        label_features = [{'input_ids': feature['labels']} for feature in features]

        batch = self.processor.pad(input_features, padding=self.padding, return_tensors='pt')
        labels_batch = self.processor.pad(labels=label_features, padding=self.padding, return_tensors='pt')

        labels = labels_batch['input_ids'].masked_fill(labels_batch.attention_mask.ne(1), -100)

        batch['labels'] = labels

        return batch


def normalize_spaces(text):
    if text is None:
        return ''

    text = str(text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)

    label_str = processor.batch_decode(label_ids, group_tokens=False)

    pred_str = [normalize_spaces(text) for text in pred_str]
    label_str = [normalize_spaces(text) for text in label_str]

    wer = jiwer.wer(label_str, pred_str)
    cer = jiwer.cer(label_str, pred_str)

    return {'wer': wer, 'cer': cer}


def print_trainable_parameters(model):
    trainable_params = sum(param.numel() for param in model.parameters() if param.requires_grad)
    all_params = sum(param.numel() for param in model.parameters())

    print(f'Trainable params: {trainable_params:,}')
    print(f'All params: {all_params:,}')
    print(f'Trainable share: {100 * trainable_params / all_params:.4f}%')


def get_best_dev_metrics(trainer, log_history_df):
    best_checkpoint = trainer.state.best_model_checkpoint
    best_dev_cer = trainer.state.best_metric

    best_step = None
    best_dev_loss = None
    best_dev_wer = None

    if best_checkpoint is not None:
        match = re.search(r'checkpoint-(\d+)', best_checkpoint)

        if match is not None:
            best_step = int(match.group(1))

            best_eval_rows = log_history_df[
                (log_history_df['step'] == best_step) &
                (log_history_df['eval_cer'].notna())
            ]

            if len(best_eval_rows) > 0:
                best_eval_row = best_eval_rows.iloc[0]
                best_dev_loss = best_eval_row['eval_loss']
                best_dev_wer = best_eval_row['eval_wer']
                best_dev_cer = best_eval_row['eval_cer']

    return {
        'best_dev_checkpoint': best_checkpoint,
        'best_dev_step': best_step,
        'best_dev_loss': best_dev_loss,
        'best_dev_WER': best_dev_wer,
        'best_dev_CER': best_dev_cer
    }


def prepare_resource_test_dataset(resource):
    test_dataset_raw_resource = dataset_dict['test'].filter(lambda example: example['resource'] == resource)

    if len(test_dataset_raw_resource) == 0:
        raise ValueError(f'No test examples found for resource: {resource}')

    test_dataset_resource = test_dataset_raw_resource.map(prepare_dataset,
        remove_columns=test_dataset_raw_resource.column_names, load_from_cache_file=False)

    decoded_labels = processor.batch_decode(test_dataset_resource['labels'], group_tokens=False)
    decoded_labels = [normalize_spaces(text) for text in decoded_labels]

    references = [normalize_spaces(text) for text in test_dataset_raw_resource['sentence']]
    mismatches = [(i, ref, label) for i, (ref, label) in enumerate(zip(references, decoded_labels)) if ref != label]

    print(f'{resource} mismatches before predict:', len(mismatches))

    if len(mismatches) > 0:
        print(mismatches[:5])
        raise ValueError(f'{resource}: raw references and prepared labels do not match')

    return test_dataset_raw_resource, test_dataset_resource


def predict_dataset_in_order(model, prepared_dataset, raw_dataset, data_collator, processor, batch_size=8):
    device = next(model.parameters()).device
    model.eval()

    rows = []

    for start in tqdm(range(0, len(prepared_dataset), batch_size)):
        end = min(start + batch_size, len(prepared_dataset))

        features = [prepared_dataset[i] for i in range(start, end)]

        batch = data_collator(features)

        input_batch = {key: value.to(device) for key, value in batch.items() if key != 'labels'}

        with torch.no_grad():
            logits = model(**input_batch).logits

        pred_ids = torch.argmax(logits, dim=-1)

        pred_str = processor.batch_decode(pred_ids)
        pred_str = [normalize_spaces(text) for text in pred_str]

        for local_i, prediction in enumerate(pred_str):
            raw_i = start + local_i
            raw_example = raw_dataset[raw_i]

            rows.append({
                'resource': raw_example['resource'],
                'path': raw_example['path'],
                'reference': normalize_spaces(raw_example['sentence']),
                'prediction': prediction
            })

    return pd.DataFrame(rows)


def evaluate_resource_test(resource, experiment_name, model, batch_size=8):
    test_dataset_raw_resource, test_dataset_resource = prepare_resource_test_dataset(resource)

    results_df = predict_dataset_in_order(model=model, prepared_dataset=test_dataset_resource,
        raw_dataset=test_dataset_raw_resource, data_collator=data_collator,
        processor=processor, batch_size=batch_size)

    wer = jiwer.wer(results_df['reference'].tolist(), results_df['prediction'].tolist())

    cer = jiwer.cer(results_df['reference'].tolist(), results_df['prediction'].tolist())

    predictions_path = f'{experiment_name}_{resource}_test_predictions.csv'

    results_df.to_csv(predictions_path, index=False, encoding='utf-8-sig')

    print(f'{resource}_test WER:', wer)
    print(f'{resource}_test CER:', cer)

    row = {
        'model': model_label,
        'training': experiment_name,
        'subset': f'{resource}_test',
        'WER': wer,
        'CER': cer,
        'n_files': len(results_df),
        'predictions_file': predictions_path
    }

    return row, results_df

In [5]:
dataset_repo_id = 'tadgeis/chukchi-asr-data-private'

dataset_dict = load_dataset(dataset_repo_id, token=hf_token)
dataset_dict = dataset_dict.cast_column('audio', Audio(sampling_rate=16_000))

dataset_dict

DatasetDict({
    train: Dataset({
        features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
        num_rows: 2714
    })
    test: Dataset({
        features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
        num_rows: 429
    })
    dev: Dataset({
        features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
        num_rows: 140
    })
})

In [6]:
train_chars = set(' '.join(dataset_dict['train']['sentence']))
dev_chars = set(' '.join(dataset_dict['dev']['sentence']))
test_chars = set(' '.join(dataset_dict['test']['sentence']))

print('Chars in dev but not train:', dev_chars - train_chars)
print('Chars in test but not train:', test_chars - train_chars)

print('Train chars:', sorted(train_chars))

Chars in dev but not train: set()
Chars in test but not train: set()
Train chars: [' ', "'", 'а', 'б', 'в', 'г', 'д', 'е', 'ж', 'з', 'и', 'й', 'к', 'л', 'м', 'н', 'о', 'п', 'р', 'с', 'т', 'у', 'ф', 'х', 'ц', 'ч', 'ш', 'щ', 'ъ', 'ы', 'ь', 'э', 'ю', 'я', 'ё', 'ӄ', 'ӈ', 'ԓ']


In [7]:
vocab_source_dataset = dataset_dict['train']

vocab_train = vocab_source_dataset.map(extract_all_chars, batched=True, batch_size=-1,
    keep_in_memory=True, remove_columns=vocab_source_dataset.column_names)

vocab_list = vocab_train['vocab'][0]
vocab_dict = {char: idx for idx, char in enumerate(vocab_list)}

vocab_dict['|'] = vocab_dict[' ']
del vocab_dict[' ']

vocab_dict['[UNK]'] = len(vocab_dict)
vocab_dict['[PAD]'] = len(vocab_dict)

Map:   0%|          | 0/2714 [00:00<?, ? examples/s]

In [8]:
with open('vocab.json', 'w', encoding='utf-8') as vocab_file:
    json.dump(vocab_dict, vocab_file, ensure_ascii=False, indent=2)

In [9]:
tokenizer = Wav2Vec2CTCTokenizer.from_pretrained('./', unk_token='[UNK]', pad_token='[PAD]', word_delimiter_token='|')

In [10]:
sample_text = dataset_dict['train'][0]['sentence']

encoded = tokenizer(sample_text).input_ids
decoded = tokenizer.decode(encoded, group_tokens=False)

print('Original:', sample_text)
print('Encoded:', encoded)
print('Decoded:', decoded)

Original: ымыԓьо кэлит тыӈивылӄылти ӄликкин январьтагнэты
Encoded: [29, 14, 29, 37, 30, 16, 0, 12, 31, 13, 10, 20, 0, 20, 29, 36, 10, 4, 29, 13, 35, 29, 13, 20, 10, 0, 35, 13, 10, 12, 12, 10, 15, 0, 33, 15, 4, 2, 18, 30, 20, 2, 5, 15, 31, 20, 29]
Decoded: ымыԓьо кэлит тыӈивылӄылти ӄликкин январьтагнэты


In [11]:
feature_extractor = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16_000,
    padding_value=0.0, do_normalize=True, return_attention_mask=True)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

In [12]:
data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

# pooled fine-tuning with eval on chuklang

In [17]:
experiment_name = 'xlsr_1b_pooled_to_chuklang'
model_id = 'facebook/wav2vec2-xls-r-1b'
model_repo_id = 'tadgeis/xlsr-1b-pooled-to-chuklang'
model_label = 'XLS-R 1B pooled CTC fine-tuning'

resources_to_evaluate = ['chuklang']

SEED = 42

In [13]:
create_repo(repo_id=model_repo_id, repo_type='model', private=False, exist_ok=True, token=hf_token)
processor.push_to_hub(model_repo_id, private=False, token=hf_token)

CommitInfo(commit_url='https://huggingface.co/tadgeis/xlsr-1b-pooled-to-chuklang/commit/20ab7e1e241bfb5ea5f170c11422cf62732bab86', commit_message='Upload processor', commit_description='', oid='20ab7e1e241bfb5ea5f170c11422cf62732bab86', pr_url=None, repo_url=RepoUrl('https://huggingface.co/tadgeis/xlsr-1b-pooled-to-chuklang', endpoint='https://huggingface.co', repo_type='model', repo_id='tadgeis/xlsr-1b-pooled-to-chuklang'), pr_revision=None, pr_num=None)

In [14]:
set_seed(SEED)

train_dataset_raw = dataset_dict['train'].filter(lambda example: example['resource'] in ['chuklang', 'radio', 'bible'])

eval_dataset_raw = dataset_dict['dev'].filter(lambda example: example['resource'] == 'chuklang')

train_dataset_raw = train_dataset_raw.shuffle(seed=SEED)

print('Train raw:', train_dataset_raw)
print('Eval raw:', eval_dataset_raw)

Filter:   0%|          | 0/2714 [00:00<?, ? examples/s]

Filter:   0%|          | 0/140 [00:00<?, ? examples/s]

Train raw: Dataset({
    features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
    num_rows: 2714
})
Eval raw: Dataset({
    features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
    num_rows: 140
})


In [15]:
train_dataset = train_dataset_raw.map(prepare_dataset, remove_columns=train_dataset_raw.column_names, load_from_cache_file=False)
eval_dataset = eval_dataset_raw.map(prepare_dataset, remove_columns=eval_dataset_raw.column_names, load_from_cache_file=False)

print('Prepared train:', train_dataset)
print('Prepared eval:', eval_dataset)
print('Train columns:', train_dataset.column_names)
print('Eval columns:', eval_dataset.column_names)

Map:   0%|          | 0/2714 [00:00<?, ? examples/s]

Map:   0%|          | 0/140 [00:00<?, ? examples/s]

Prepared train: Dataset({
    features: ['input_values', 'input_length', 'labels'],
    num_rows: 2714
})
Prepared eval: Dataset({
    features: ['input_values', 'input_length', 'labels'],
    num_rows: 140
})
Train columns: ['input_values', 'input_length', 'labels']
Eval columns: ['input_values', 'input_length', 'labels']


In [16]:
model = create_xlsr_model(model_id)

print_trainable_parameters(model)

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-xls-r-1b and are newly initialized: ['lm_head.bias', 'lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable params: 958,341,034
All params: 962,551,210
Trainable share: 99.5626%


In [17]:
# # test on the longest audio files

# def test_longest_batch(model, train_dataset, data_collator, batch_size=4):
#     if not torch.cuda.is_available():
#         raise RuntimeError('CUDA is not available.')

#     model.to('cuda')
#     model.train()
#     model.zero_grad(set_to_none=True)

#     outputs = None
#     loss = None
#     batch = None
#     features = None

#     longest_indices = np.argsort(
#         -np.array(train_dataset['input_length'])
#     )[:batch_size]

#     print('Testing indices:', longest_indices.tolist())
#     print(
#         'Lengths in seconds:',
#         [
#             round(train_dataset[int(i)]['input_length'] / 16000, 2)
#             for i in longest_indices
#         ]
#     )

#     try:
#         features = [
#             train_dataset[int(i)]
#             for i in longest_indices
#         ]

#         batch = data_collator(features)

#         batch = {
#             key: value.to('cuda')
#             for key, value in batch.items()
#         }

#         with torch.autocast(
#             device_type='cuda',
#             dtype=torch.float16
#         ):
#             outputs = model(**batch)
#             loss = outputs.loss

#         loss.backward()

#         print('Loss:', loss.item())
#         print('Longest-batch train forward/backward: PASSED')

#         return True

#     except torch.cuda.OutOfMemoryError as error:
#         print('CUDA OOM: batch_size is too large for this GPU/model/audio length.')
#         print(error)

#         return False

#     finally:
#         model.zero_grad(set_to_none=True)

#         del batch
#         del features
#         del outputs
#         del loss

#         gc.collect()

#         if torch.cuda.is_available():
#             torch.cuda.empty_cache()
#             torch.cuda.ipc_collect()

#         !nvidia-smi


# test_longest_batch(model=model, train_dataset=train_dataset, data_collator=data_collator, batch_size=16) ## batch size

In [18]:
training_args = TrainingArguments(
    output_dir=model_repo_id.split('/')[-1],

    group_by_length=True,

    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    per_device_eval_batch_size=8,

    eval_strategy='steps',
    save_strategy='steps',

    num_train_epochs=8,

    gradient_checkpointing=False,
    fp16=torch.cuda.is_available(),

    save_steps=50,
    eval_steps=50,
    logging_steps=50,

    learning_rate=3e-4, 
    lr_scheduler_type='linear',
    warmup_steps=50,

    save_total_limit=3,

    load_best_model_at_end=True,
    metric_for_best_model='cer',
    greater_is_better=False,

    push_to_hub=True,
    hub_model_id=model_repo_id,
    hub_private_repo=False,
    hub_token=hf_token,
    hub_strategy='checkpoint',
    hub_always_push=True,

    report_to='none',
    disable_tqdm=False,

    seed=SEED,
    data_seed=SEED
)

In [19]:
trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=processor.feature_extractor,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=6,
            early_stopping_threshold=0.001
        )
    ]
)

In [20]:
!df -h

Filesystem      Size  Used Avail Use% Mounted on
overlay         200G  112G   89G  56% /
tmpfs            64M     0   64M   0% /dev
shm             157G  4.0K  157G   1% /dev/shm
/dev/md0        7.0T  2.0T  5.1T  29% /etc/hosts
tmpfs           126G  1.6M  126G   1% /run/nvidia-persistenced/socket
/dev/root       124G   11G  114G   9% /usr/bin/nvidia-smi
tmpfs           315G     0  315G   0% /proc/acpi
tmpfs           315G     0  315G   0% /proc/scsi
tmpfs           315G     0  315G   0% /sys/firmware


In [21]:
trainer.train()

Step,Training Loss,Validation Loss,Wer,Cer
50,4.509000,2.876277,1.000000,0.798793
100,1.616300,2.048320,1.000000,0.478142
150,1.139700,1.892028,0.998413,0.548564
200,1.162800,1.646323,0.996825,0.445766
250,0.917600,1.648184,1.000000,0.455094
300,0.914800,1.626156,0.982540,0.398756
350,0.869800,1.316993,0.955556,0.340406
400,0.786800,1.290811,0.952381,0.341504
450,0.699100,1.233163,0.938095,0.327785
500,0.719300,1.253273,0.976190,0.363819


No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


TrainOutput(global_step=2720, training_loss=0.5382136839277604, metrics={'train_runtime': 1805.9184, 'train_samples_per_second': 12.023, 'train_steps_per_second': 1.506, 'total_flos': 1.6337486212690993e+19, 'train_loss': 0.5382136839277604, 'epoch': 8.0})

In [22]:
log_history_df = pd.DataFrame(trainer.state.log_history)

log_history_df.to_csv(f'{experiment_name}_log_history.csv', index=False, encoding='utf-8-sig')

best_metrics = get_best_dev_metrics(trainer, log_history_df)

print(best_metrics)

{'best_dev_checkpoint': 'xlsr-1b-pooled-to-chuklang/checkpoint-2700', 'best_dev_step': 2700, 'best_dev_loss': np.float64(0.9440882802009583), 'best_dev_WER': np.float64(0.746031746031746), 'best_dev_CER': np.float64(0.1946222791293214)}


In [23]:
processor.save_pretrained(training_args.output_dir)
trainer.save_model(training_args.output_dir)

trainer.push_to_hub()

processor.push_to_hub(model_repo_id, private=False, token=hf_token)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


README.md: 0.00B [00:00, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/tadgeis/xlsr-1b-pooled-to-chuklang/commit/a08167b60b455db251303b6a67c3e812a57766d1', commit_message='Upload processor', commit_description='', oid='a08167b60b455db251303b6a67c3e812a57766d1', pr_url=None, repo_url=RepoUrl('https://huggingface.co/tadgeis/xlsr-1b-pooled-to-chuklang', endpoint='https://huggingface.co', repo_type='model', repo_id='tadgeis/xlsr-1b-pooled-to-chuklang'), pr_revision=None, pr_num=None)

In [24]:
summary_rows = []
prediction_dfs = {}

for resource in resources_to_evaluate:
    row, results_df = evaluate_resource_test(resource=resource, experiment_name=experiment_name,
        model=trainer.model, batch_size=16) ### batch size

    row['best_dev_checkpoint'] = best_metrics['best_dev_checkpoint']
    row['best_dev_step'] = best_metrics['best_dev_step']
    row['best_dev_loss'] = best_metrics['best_dev_loss']
    row['best_dev_WER'] = best_metrics['best_dev_WER']
    row['best_dev_CER'] = best_metrics['best_dev_CER']

    summary_rows.append(row)
    prediction_dfs[resource] = results_df

summary_df = pd.DataFrame(summary_rows)

summary_df.to_csv(f'{experiment_name}_results_summary.csv', index=False, encoding='utf-8-sig')

summary_df

Map: 100%|##########| 200/200 [00:00<?, ? examples/s]

chuklang mismatches before predict: 0


  0%|          | 0/13 [00:00<?, ?it/s]

chuklang_test WER: 0.7685076380728555
chuklang_test CER: 0.20682037829628608


,model,training,subset,WER,CER,n_files,predictions_file,best_dev_checkpoint,best_dev_step,best_dev_loss,best_dev_WER,best_dev_CER
0,XLS-R 1B pooled CTC fine-tuning,xlsr_1b_pooled_to_chuklang,chuklang_test,0.768508,0.20682,200,xlsr_1b_pooled_to_chuklang_chuklang_test_predi...,xlsr-1b-pooled-to-chuklang/checkpoint-2700,2700,0.944088,0.746032,0.194622


In [27]:
prediction_dfs['chuklang'].head()

,resource,path,reference,prediction
0,chuklang,A chatterbox and a wanton girl_2.wav,ӄоле итгъэт ӄынвэтэ ӈиръэ ӈэвысӄэтти элерэты н...,ӄоле итгъэт ӄынвэтэ ӈиръэӈ ӈэвысӄыти лерэтынат...
1,chuklang,A chatterbox and a wanton girl_3.wav,ӄол вэтгавӈавъым ӄол камэлгыӈав,ӄолы вэтъгэвӈаъым ӄол камэтлыӈа
2,chuklang,A chatterbox and a wanton girl_4.wav,ынкы илирыкы нантыӈӈонатъым,ынкы илирэк ӈантыӈӈонатъым
3,chuklang,Abramovich_4.wav,гэчевкы нынтыӄин таӈколё ынкы ныгынритӄин,ачевэ нынтыӄин таӈколёӈкы ныгынмэтӄин
4,chuklang,An evil spirit and a dicky bird_1.wav,энмэн гатвален каԓьайӈын ынкъам пчеӄалгын,эмэ гатвален каԓьай ынкъам пчеӄыл


In [28]:
files_to_download = [Path(f'{experiment_name}_log_history.csv'),
                     Path(f'{experiment_name}_results_summary.csv')]

for resource in resources_to_evaluate:
    files_to_download.append(Path(f'{experiment_name}_{resource}_test_predictions.csv'))

for path in files_to_download:
    if path.exists():
        print(f'Ready to download: {path}')
    else:
        print(f'File not found: {path}')

Ready to download: xlsr_1b_pooled_to_chuklang_log_history.csv
Ready to download: xlsr_1b_pooled_to_chuklang_results_summary.csv
Ready to download: xlsr_1b_pooled_to_chuklang_chuklang_test_predictions.csv


# Stage 2: chuklang-only fine-tuning after pooled training

In [18]:
stage2_experiment_name = f'{experiment_name}_stage2_chuklang_only'
stage2_model_repo_id = f'{model_repo_id}-stage2-chuklang'
stage2_model_label = f'{model_label} + chuklang-only Stage 2'

stage2_resources_to_evaluate = ['chuklang']

create_repo(
    repo_id=stage2_model_repo_id,
    repo_type='model',
    private=False,
    exist_ok=True,
    token=hf_token
)

processor.push_to_hub(stage2_model_repo_id, private=False, token=hf_token)

print('Stage 2 experiment:', stage2_experiment_name)
print('Stage 2 repo:', stage2_model_repo_id)

Stage 2 experiment: xlsr_1b_pooled_to_chuklang_stage2_chuklang_only
Stage 2 repo: tadgeis/xlsr-1b-pooled-to-chuklang-stage2-chuklang


In [19]:
stage2_train_dataset_raw = dataset_dict['train'].filter(
    lambda example: example['resource'] == 'chuklang'
)

stage2_eval_dataset_raw = dataset_dict['dev'].filter(
    lambda example: example['resource'] == 'chuklang'
)

stage2_train_dataset_raw = stage2_train_dataset_raw.shuffle(seed=SEED)

print('Stage 2 train raw:', stage2_train_dataset_raw)
print('Stage 2 eval raw:', stage2_eval_dataset_raw)

Stage 2 train raw: Dataset({
    features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
    num_rows: 659
})
Stage 2 eval raw: Dataset({
    features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
    num_rows: 140
})


In [20]:
stage2_train_dataset = stage2_train_dataset_raw.map(
    prepare_dataset,
    remove_columns=stage2_train_dataset_raw.column_names,
    load_from_cache_file=False
)

stage2_eval_dataset = stage2_eval_dataset_raw.map(
    prepare_dataset,
    remove_columns=stage2_eval_dataset_raw.column_names,
    load_from_cache_file=False
)

print('Stage 2 prepared train:', stage2_train_dataset)
print('Stage 2 prepared eval:', stage2_eval_dataset)

Map: 100%|##########| 659/659 [00:00<?, ? examples/s]

Map: 100%|##########| 140/140 [00:00<?, ? examples/s]

Stage 2 prepared train: Dataset({
    features: ['input_values', 'input_length', 'labels'],
    num_rows: 659
})
Stage 2 prepared eval: Dataset({
    features: ['input_values', 'input_length', 'labels'],
    num_rows: 140
})


In [21]:
pooled_checkpoint = model_repo_id

processor = Wav2Vec2Processor.from_pretrained(pooled_checkpoint, token=hf_token)

stage2_model = Wav2Vec2ForCTC.from_pretrained(
    pooled_checkpoint,
    token=hf_token
)

stage2_model.freeze_feature_encoder()

print_trainable_parameters(stage2_model)

Trainable params: 958,341,034
All params: 962,551,210
Trainable share: 99.5626%


In [22]:
stage2_training_args = TrainingArguments(
    output_dir=stage2_model_repo_id.split('/')[-1],

    group_by_length=True,

    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    per_device_eval_batch_size=8,

    eval_strategy='steps',
    save_strategy='steps',

    num_train_epochs=8,

    gradient_checkpointing=False,
    fp16=torch.cuda.is_available(),

    save_steps=25,
    eval_steps=25,
    logging_steps=25,

    learning_rate=1e-5,
    lr_scheduler_type='linear',
    warmup_steps=0,

    save_total_limit=3,

    load_best_model_at_end=True,
    metric_for_best_model='cer',
    greater_is_better=False,

    push_to_hub=True,
    hub_model_id=stage2_model_repo_id,
    hub_private_repo=False,
    hub_token=hf_token,
    hub_strategy='checkpoint',
    hub_always_push=True,

    report_to='none',
    disable_tqdm=False,

    seed=SEED,
    data_seed=SEED
)

In [23]:
!rm -rf "{stage2_training_args.output_dir}"
!rm -rf "{training_args.output_dir}"

In [24]:
stage2_trainer = Trainer(
    model=stage2_model,
    data_collator=data_collator,
    args=stage2_training_args,
    compute_metrics=compute_metrics,
    train_dataset=stage2_train_dataset,
    eval_dataset=stage2_eval_dataset,
    processing_class=processor.feature_extractor,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=6,
            early_stopping_threshold=0.001
        )
    ]
)

In [25]:
stage2_trainer.train()

Step,Training Loss,Validation Loss,Wer,Cer
25,0.185500,0.940633,0.752381,0.196086
50,0.227000,0.946388,0.750794,0.196634
75,0.225300,0.947329,0.747619,0.194988
100,0.186300,0.952856,0.739683,0.192610
125,0.231400,0.954441,0.742857,0.190964
150,0.183300,0.960437,0.747619,0.193525
175,0.191000,0.952955,0.749206,0.193159
200,0.177400,0.953108,0.741270,0.190781
225,0.222500,0.961637,0.747619,0.192976
250,0.180400,0.967542,0.746032,0.193891


No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


TrainOutput(global_step=275, training_loss=0.1959648340398615, metrics={'train_runtime': 224.5269, 'train_samples_per_second': 23.48, 'train_steps_per_second': 2.957, 'total_flos': 1.0584875032376092e+18, 'train_loss': 0.1959648340398615, 'epoch': 3.3132530120481927})

In [26]:
stage2_log_history_df = pd.DataFrame(stage2_trainer.state.log_history)

stage2_log_history_df.to_csv(
    f'{stage2_experiment_name}_log_history.csv',
    index=False,
    encoding='utf-8-sig'
)

stage2_best_metrics = get_best_dev_metrics(stage2_trainer, stage2_log_history_df)

print(stage2_best_metrics)

{'best_dev_checkpoint': 'xlsr-1b-pooled-to-chuklang-stage2-chuklang/checkpoint-200', 'best_dev_step': 200, 'best_dev_loss': np.float64(0.9531084299087524), 'best_dev_WER': np.float64(0.7412698412698413), 'best_dev_CER': np.float64(0.19078104993597952)}


In [27]:
processor.save_pretrained(stage2_training_args.output_dir)
stage2_trainer.save_model(stage2_training_args.output_dir)

stage2_trainer.push_to_hub()

processor.push_to_hub(stage2_model_repo_id, private=False, token=hf_token)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


README.md: 0.00B [00:00, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/tadgeis/xlsr-1b-pooled-to-chuklang-stage2-chuklang/commit/dbb319370f6625f86325a004e888ea54b99b93d8', commit_message='Upload processor', commit_description='', oid='dbb319370f6625f86325a004e888ea54b99b93d8', pr_url=None, repo_url=RepoUrl('https://huggingface.co/tadgeis/xlsr-1b-pooled-to-chuklang-stage2-chuklang', endpoint='https://huggingface.co', repo_type='model', repo_id='tadgeis/xlsr-1b-pooled-to-chuklang-stage2-chuklang'), pr_revision=None, pr_num=None)

In [28]:
stage2_summary_rows = []
stage2_prediction_dfs = {}

for resource in stage2_resources_to_evaluate:
    row, results_df = evaluate_resource_test(
        resource=resource,
        experiment_name=stage2_experiment_name,
        model=stage2_trainer.model,
        batch_size=4
    )

    row['model'] = stage2_model_label
    row['best_dev_checkpoint'] = stage2_best_metrics['best_dev_checkpoint']
    row['best_dev_step'] = stage2_best_metrics['best_dev_step']
    row['best_dev_loss'] = stage2_best_metrics['best_dev_loss']
    row['best_dev_WER'] = stage2_best_metrics['best_dev_WER']
    row['best_dev_CER'] = stage2_best_metrics['best_dev_CER']

    stage2_summary_rows.append(row)
    stage2_prediction_dfs[resource] = results_df

stage2_summary_df = pd.DataFrame(stage2_summary_rows)

stage2_summary_df.to_csv(
    f'{stage2_experiment_name}_results_summary.csv',
    index=False,
    encoding='utf-8-sig'
)

stage2_summary_df

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

chuklang mismatches before predict: 0


  0%|          | 0/50 [00:00<?, ?it/s]

chuklang_test WER: 0.7779083431257344
chuklang_test CER: 0.20543973491647108


,model,training,subset,WER,CER,n_files,predictions_file,best_dev_checkpoint,best_dev_step,best_dev_loss,best_dev_WER,best_dev_CER
0,XLS-R 1B pooled CTC fine-tuning + chuklang-onl...,xlsr_1b_pooled_to_chuklang_stage2_chuklang_only,chuklang_test,0.777908,0.20544,200,xlsr_1b_pooled_to_chuklang_stage2_chuklang_onl...,xlsr-1b-pooled-to-chuklang-stage2-chuklang/che...,200,0.953108,0.74127,0.190781


In [29]:
stage2_prediction_dfs['chuklang'].head()

,resource,path,reference,prediction
0,chuklang,A chatterbox and a wanton girl_2.wav,ӄоле итгъэт ӄынвэтэ ӈиръэ ӈэвысӄэтти элерэты н...,ӄоле итгъэт ӄынвэтэ ӈиръэӈ ӈэвысӄыти лерэтынат...
1,chuklang,A chatterbox and a wanton girl_3.wav,ӄол вэтгавӈавъым ӄол камэлгыӈав,ӄол вэтгавӈаъы'м ӄол камэтлгыӈа
2,chuklang,A chatterbox and a wanton girl_4.wav,ынкы илирыкы нантыӈӈонатъым,ынкы илирэк ӈантыӈӈонатъым
3,chuklang,Abramovich_4.wav,гэчевкы нынтыӄин таӈколё ынкы ныгынритӄин,ачевэ нынтыӄин таӈколёӈкы ныгынмитӄин
4,chuklang,An evil spirit and a dicky bird_1.wav,энмэн гатвален каԓьайӈын ынкъам пчеӄалгын,эмэ гатвален каԓьай ынкъам пчеӄыл


In [30]:
stage2_files_to_download = [
    Path(f'{stage2_experiment_name}_log_history.csv'),
    Path(f'{stage2_experiment_name}_results_summary.csv')
]

for resource in stage2_resources_to_evaluate:
    stage2_files_to_download.append(
        Path(f'{stage2_experiment_name}_{resource}_test_predictions.csv')
    )

for path in stage2_files_to_download:
    if path.exists():
        print(f'Ready to download: {path}')
    else:
        print(f'File not found: {path}')

Ready to download: xlsr_1b_pooled_to_chuklang_stage2_chuklang_only_log_history.csv
Ready to download: xlsr_1b_pooled_to_chuklang_stage2_chuklang_only_results_summary.csv
Ready to download: xlsr_1b_pooled_to_chuklang_stage2_chuklang_only_chuklang_test_predictions.csv
